# 51 조건 추출 반복과 검색 범위 검사

개발 질문 40개를 3회 새로 추출한다. 1회차만 다음 비교에 사용하고 2·3회차는 안정성 검사에만 쓴다. 조회 전 의미 검증으로 정상 조건을 거절하지 않는다. 카드·발급사 범위는 질문에서 독립적으로 식별한다.

공유 도우미는 장시간 API의 요청 ID·원응답 저장·비용 예약·재개 검증·시간 계측을 동일하게 유지하기 위해 재사용한다. 첫 토큰·브라우저 표시 시간은 측정하지 않는다. 아래 전체 셀을 순서대로 실행하며 PAID_RETEST_ALLOW_NETWORK=1일 때만 새 API 호출을 허용한다.

In [1]:
from pathlib import Path
import importlib.util,sys,json,os,time,resource
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
spec=importlib.util.spec_from_file_location('runtime51',ROOT/'notebooks/scripts/51_experiment_runtime.py')
X=importlib.util.module_from_spec(spec);sys.modules[spec.name]=X;spec.loader.exec_module(X)
P=X.P;Q=X.Q;PATH=P.F.STAGES[51];PATH.mkdir(exist_ok=True);P.F.initialize(51)
assert P.F.read_json(P.F.STAGES[50]/'completion_status.json')['paid_stage_allowed']
queries=P.F.read_json(P.F.STAGES[50]/'questions.json')['queries']
X.frozen(PATH/'questions.json',dict(queries=queries,split='development'))
X.frozen(PATH/'parser_prompt_schema.json',dict(prompt=Q.PROMPT,schema=Q.QueryPlan.model_json_schema()))
ids=[f"{q['query_id']}:parser:r{r}" for r in range(1,4) for q in queries]
manifest=X.make_manifest(51,ROOT/'notebooks/51_query_extraction_scope.ipynb',queries,
 dict(parser=ids,answer=[]),maximum_parser=120,stage_cap='0.4',extra=dict(
 split='development',next_stage_parser_repeat=1,logical_parser_conditions=120,
 interpretation='repeat2/3 stability only; do not replace repeat1 with a better plan',
 first_token_seconds=None,browser_display_seconds=None))
X.guard_manifest(manifest)
ALLOW_NETWORK=os.getenv('PAID_RETEST_ALLOW_NETWORK')=='1'
print('동결한 추출 조건',len(ids),'전체 누적비용',X.ledger())

동결한 추출 조건 120 전체 누적비용 {'record_count': 376, 'known_priced_records': 375, 'unknown_cost_records': 1, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.253103230', 'tokens': {'input_tokens': 1803035, 'output_tokens': 276152, 'ordinary_input_tokens': 57316, 'cached_input_tokens': 948759, 'cache_write_tokens': 796960, 'embedding_tokens': 9402}, 'billed_cost_usd': None}


In [2]:
started=time.perf_counter();cpu_started=time.process_time();records=[]
for repeat in range(1,4):
 for q in queries:
  record=X.extract(51,q,repeat,allow_network=ALLOW_NETWORK);records.append(record)
  P.F.write_json(PATH/'progress.json',dict(completed=len(records),expected=120,last_identity=record['parser']['identity'],ledger=X.ledger()))
  print(record['parser']['identity'],record['parser']['status'],record['routing']['reason'],len(record['routing']['eligible_cards']),round(record['parser']['parser_wall_seconds'],3),'초',flush=True)
P.F.write_json(PATH/'execution_results.json',dict(records=records,manifest_sha256=P.sha(manifest)))
# 1회차는 후속 단계에서 바꿔 끼우지 못하도록 별도 봉인
X.frozen(PATH/'repeat1_plans.json',dict(records=[r for r in records if r['parser']['repeat']==1],manifest_sha256=P.sha(manifest)))
print('추출 완료',len(records),'누적비용',X.ledger())

H01:parser:r1 ok no_results 0 7.703 초
H02:parser:r1 ok matched 1 3.975 초
H03:parser:r1 ok matched 1 4.434 초
H04:parser:r1 ok no_results 0 8.18 초
H05:parser:r1 ok matched 1 6.668 초
H06:parser:r1 ok no_results 0 4.458 초
H07:parser:r1 ok no_results 0 4.222 초
H08:parser:r1 ok matched 1 5.195 초
H09:parser:r1 ok no_results 0 4.678 초
H10:parser:r1 ok no_results 0 9.339 초
H11:parser:r1 ok matched 5 4.309 초
H12:parser:r1 ok no_results 0 5.127 초
H13:parser:r1 ok no_results 0 4.786 초
H14:parser:r1 ok matched 1 7.499 초
H15:parser:r1 ok matched 2 6.736 초
H16:parser:r1 ok no_results 0 4.976 초
H17:parser:r1 ok no_results 0 4.293 초
H18:parser:r1 ok matched 6 3.83 초
H19:parser:r1 ok no_results 0 9.3 초
H20:parser:r1 ok no_results 0 3.856 초
H21:parser:r1 ok no_results 0 6.397 초
H22:parser:r1 ok matched 1 3.993 초
H23:parser:r1 ok matched 1 5.097 초
H24:parser:r1 ok no_results 0 6.101 초
H25:parser:r1 ok ambiguous_or_unknown_product 0 4.01 초
H26:parser:r1 ok ambiguous_or_unknown_product 0 4.823 초
H27:parser:

In [3]:
# 실제 운영 Vector/FTS 검색기의 사전 범위 제한만 검증한다.
# 쿼리 벡터는 기존 첫 문서 벡터를 사용한 경계 검사이며 검색 품질 측정은 아니다.
import numpy as np
from pickcardu_rag.retrieval import InMemorySquaredL2Searcher
from pickcardu_rag_api.index import SQLiteFTSSearcher
checks=[]
for profile in X.PROFILES:
 source=P.DATA/'31_operational_chunking_comparison'
 chunks=[json.loads(l) for l in (source/(profile+'_chunks.jsonl')).read_text().splitlines()]
 vectors=np.load(source/profile/'embeddings.npy',mmap_mode='r',allow_pickle=False)
 dense=InMemorySquaredL2Searcher([c['chunk_id'] for c in chunks],vectors,card_keys=[c['document_id'] for c in chunks])
 lexical=SQLiteFTSSearcher(source/profile/'corpus.sqlite')
 owners={c['chunk_id']:c['document_id'] for c in chunks}
 for record in records:
  scope=record['routing']['scope']
  if scope is None or record['routing']['route']=='clarify':continue
  vec=dense.search(vectors[0],limit=50,card_keys=tuple(scope))
  bm=lexical.search(record['parser']['question'],limit=50,card_keys=tuple(scope))
  leaked=[r.chunk_id for r in vec+bm if owners[r.chunk_id] not in scope]
  expected=min(50,sum(c['document_id'] in scope for c in chunks))
  assert len(vec)==expected and not leaked
  checks.append(dict(identity=record['parser']['identity'],profile=profile,scope=scope,vector_count=len(vec),bm25_count=len(bm),out_of_scope=leaked))
P.F.write_json(PATH/'scope_checks.json',dict(records=checks,synthetic_boundary_probe=True,quality_evaluation=False))
summary=[]
for repeat in range(1,4):
 rows=[r for r in records if r['parser']['repeat']==repeat]
 summary.append(dict(repeat=repeat,questions=len(rows),parser_ok=sum(r['parser']['status']=='ok' for r in rows),
  db_attempted=sum(r['routing']['db_attempted'] for r in rows),matched=sum(r['routing']['reason']=='matched' for r in rows),
  no_results=sum(r['routing']['reason']=='no_results' for r in rows),clarify=sum(r['routing']['route']=='clarify' for r in rows),
  parser_seconds=X.timing_summary(r['parser']['parser_wall_seconds'] for r in rows),
  db_seconds=X.timing_summary(r['routing'].get('db_wall_seconds') for r in rows)))
stability=[]
for q in queries:
 rows=[r for r in records if r['parser']['query_id']==q['query_id']]
 signatures=[P.sha(dict(plan=r['parser']['plan'],cards=r['routing']['eligible_cards'],reason=r['routing']['reason'])) for r in rows]
 stability.append(dict(query_id=q['query_id'],distinct_plan_route_outputs=len(set(signatures)),signatures=signatures))
P.F.write_json(PATH/'summary.json',dict(by_repeat=summary,stability=stability,ledger=X.ledger(),scope_probe_cases=len(checks),out_of_scope=0,
 wall_seconds=time.perf_counter()-started,cpu_seconds=time.process_time()-cpu_started,peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
 answer_output_seconds=None,first_token_seconds=None,browser_display_seconds=None,
 time_note='추출과 DB 단계 실측. 최종 답변 생성 없음. 저장된 parser_wall_seconds는 최초 논리 실행이며 캐시 재생 시간이 아님.'))
passed=len(records)==120 and len(checks)>0 and all(s['parser_ok']==40 for s in summary)
P.F.write_json(PATH/'completion_status.json',dict(status='awaiting_semantic_review' if passed else 'technical_failure',technical_passed=passed,stage52_allowed=False,manifest_sha256=P.sha(manifest)))
print(json.dumps(summary,ensure_ascii=False,indent=2));print('범위 위반 0, 의미 검토 전 다음 유료 단계 진입 금지')

[
  {
    "repeat": 1,
    "questions": 40,
    "parser_ok": 39,
    "db_attempted": 37,
    "matched": 19,
    "no_results": 18,
    "clarify": 2,
    "parser_seconds": {
      "n": 40,
      "mean": 5.282724527874234,
      "median": 4.8043381969982875,
      "p95": 8.585025836990098,
      "maximum": 9.339238394997665
    },
    "db_seconds": {
      "n": 37,
      "mean": 0.04305617640902464,
      "median": 0.04853863301104866,
      "p95": 0.0971459779975703,
      "maximum": 0.40824763699492905
    }
  },
  {
    "repeat": 2,
    "questions": 40,
    "parser_ok": 38,
    "db_attempted": 36,
    "matched": 20,
    "no_results": 16,
    "clarify": 2,
    "parser_seconds": {
      "n": 40,
      "mean": 6.796088199899532,
      "median": 5.157521979497687,
      "p95": 12.906984874003683,
      "maximum": 31.097079422994284
    },
    "db_seconds": {
      "n": 36,
      "mean": 0.04497398016812642,
      "median": 0.05033927249314729,
      "p95": 0.09628992399666458,
      "maxim

## 승인된 HTTP 520 재개

원래 57개 행과 manifest·코드 셀은 보존했다. `scripts/51_budget_recovery.py` 및 `data/51_query_extraction_scope/budget_recovery_contract.json`이 사용자 지시에 따른 예외를 기록한다. 재개 엔진은 첫 코드 셀 뒤 `recovery.install(X)`를 실행하고 남은 원래 코드 셀을 실행한다. 최초 실패 요청은 미확인 비용으로 남기고 최대 예약액을 차감한다. 별도 1회 진단 재시도는 `http520_recovery_result.json`에 저장하며, 원래 반복 행이나 1회차 조건을 대체하지 않는다. 독립 리뷰와 비용 예외 회귀6개를 통과했다. 이 확장 없이 원래 노트북을 단독 재생하면 기존 비용 미확인 보호 장치가 중단시키는 것이 정상이다.